# Task Decomposition in Prompts Tutorial

## Overview

This tutorial explores the concept of task decomposition in prompt engineering, focusing on techniques for breaking down complex tasks and chaining subtasks in prompts. These techniques are essential for effectively leveraging large language models to solve multi-step problems and perform complex reasoning tasks.

## Motivation

As open-source language models become more capable, they can handle increasingly complex tasks — especially when given clear, step-by-step instructions. Task decomposition is a powerful technique that allows us to break down complex problems into smaller, more manageable subtasks. This approach not only improves the model's performance but also enhances the interpretability and reliability of the results.

## Key Components

1. **Breaking Down Complex Tasks**: Techniques for analyzing and dividing complex problems into simpler subtasks.
2. **Chaining Subtasks**: Methods for sequentially connecting multiple subtasks to solve a larger problem.
3. **Prompt Design for Subtasks**: Crafting effective prompts for each decomposed subtask.
4. **Result Integration**: Combining the outputs from individual subtasks to form a comprehensive solution.

## Method Details

The tutorial employs a step-by-step approach to demonstrate task decomposition:

1. **Problem Analysis**: We start by examining a complex task and identifying its component parts.
2. **Subtask Definition**: We define clear, manageable subtasks that collectively address the main problem.
3. **Prompt Engineering**: For each subtask, we create targeted prompts that guide the model.
4. **Sequential Execution**: We implement a chain of prompts, where the output of one subtask feeds into the next.
5. **Result Synthesis**: Finally, we combine the outputs from all subtasks to form a comprehensive solution.

Throughout the tutorial, we use practical examples to illustrate these concepts, demonstrating how task decomposition can be applied to various domains such as analysis, problem-solving, and creative tasks.

## Implementation Notes

This notebook runs entirely on **Google Colab** using the open-source **Qwen3-8B** model via HuggingFace Transformers. No API keys or paid services are required — the model runs locally on the Colab GPU with 4-bit quantization.

## Conclusion

By the end of this tutorial, learners will have gained practical skills in:
- Analyzing complex tasks and breaking them down into manageable subtasks
- Designing effective prompts for each subtask
- Chaining prompts to guide a local language model through a multi-step reasoning process
- Integrating results from multiple subtasks to solve complex problems

These skills will enable more effective use of open-source language models for complex problem-solving and enhance the overall quality and reliability of AI-assisted tasks.

## Setup

First, let's install dependencies, load the model, and define our helper functions. We use **Qwen3-8B** with 4-bit quantization so it fits in a free Colab GPU.

In [ ]:
# --- Google Colab Setup ---
!pip install -q transformers>=4.51.0 accelerate bitsandbytes torch

import torch
from google.colab import drive
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

drive.mount('/content/drive')
CACHE_DIR = "/content/drive/MyDrive/models"
MODEL_NAME = "Qwen/Qwen3-8B"

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4",
)
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, cache_dir=CACHE_DIR)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, device_map="auto", quantization_config=quantization_config,
    cache_dir=CACHE_DIR, torch_dtype="auto",
)

def generate(messages, max_new_tokens=512, temperature=0.7, do_sample=True):
    """Generate a response from the model given a list of chat messages."""
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer([text], return_tensors="pt").to(model.device)
    output_ids = model.generate(
        **inputs, max_new_tokens=max_new_tokens,
        temperature=temperature if do_sample else None, do_sample=do_sample, top_k=20,
    )
    generated = output_ids[0][inputs.input_ids.shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True)

def run_subtask(system_prompt, user_prompt, context=None, **kwargs):
    """Run a subtask, optionally with context from previous steps."""
    messages = [{"role": "system", "content": system_prompt}]
    if context:
        user_prompt = f"Previous analysis:\n{context}\n\nNow: {user_prompt}"
    messages.append({"role": "user", "content": user_prompt})
    return generate(messages, **kwargs)

## Why Decomposition Works: Reducing the Search Space

When an LLM generates text, each token is sampled from a probability distribution. Even a strong model doesn't achieve 100% accuracy on every token — there is always some small probability of producing a suboptimal or incorrect token. For a single token the chance of being "right" might be, say, 95%. That sounds high, but over a long generation the probabilities **multiply**:

| Sequence length | P(all correct) at 95%/token | P(all correct) at 98%/token |
|---|---|---|
| 50 tokens | 0.95^50 ≈ 0.08 (8%) | 0.98^50 ≈ 0.36 (36%) |
| 200 tokens | 0.95^200 ≈ 0.00004 | 0.98^200 ≈ 0.02 (2%) |
| 500 tokens | ≈ 0 | ≈ 0 |

A single prompt asking for a 500-token analysis has an astronomically small chance of getting *everything* right. But if we decompose into 5 subtasks of 100 tokens each, each subtask has a much higher probability of being correct, and **we can verify each step before proceeding**.

> **Key insight**: Decomposition doesn't just help organization — it fundamentally changes the probability of getting a correct result by reducing the length of each generation.


In [ ]:
# Compare: complex analysis in one shot vs. broken into 3 focused steps

project_desc = """An e-commerce REST API built with Python Flask that handles
user authentication via JWT, product catalog with search, shopping cart management,
and payment processing through Stripe. Uses PostgreSQL with SQLAlchemy ORM and
Redis for session caching. Currently serves ~10,000 requests/minute."""

# --- ONE-SHOT: Ask for everything at once ---
one_shot_prompt = f"""Analyze the following software project and provide:
1. Architecture assessment (design patterns, modularity, coupling)
2. Security vulnerabilities (injection risks, auth weaknesses, data exposure)
3. Performance bottlenecks (algorithmic complexity, memory leaks, I/O issues)
4. Recommended improvements with priority ranking

Project: {project_desc}"""

one_shot_result = generate([
    {"role": "system", "content": "You are a senior software architect conducting a thorough code review."},
    {"role": "user", "content": one_shot_prompt},
], max_new_tokens=1024)

print("=" * 60)
print("ONE-SHOT RESULT (single prompt, up to 1024 tokens)")
print("=" * 60)
print(one_shot_result)

# --- DECOMPOSED: Three focused steps ---
step1 = run_subtask(
    "You are a software architect. Focus ONLY on architecture.",
    f"Assess the architecture of this project. Evaluate design patterns, modularity, and coupling.\n\nProject: {project_desc}",
    max_new_tokens=300,
)
print("\n" + "=" * 60)
print("DECOMPOSED STEP 1: Architecture Assessment (~300 tokens)")
print("=" * 60)
print(step1)

step2 = run_subtask(
    "You are a security engineer. Focus ONLY on security vulnerabilities.",
    f"Identify security vulnerabilities in this project.\n\nProject: {project_desc}",
    context=step1,
    max_new_tokens=300,
)
print("\n" + "=" * 60)
print("DECOMPOSED STEP 2: Security Assessment (~300 tokens)")
print("=" * 60)
print(step2)

step3 = run_subtask(
    "You are a tech lead synthesizing a review.",
    f"Based on the architecture and security analyses, provide prioritized improvements.\n\nProject: {project_desc}",
    context=f"Architecture:\n{step1}\n\nSecurity:\n{step2}",
    max_new_tokens=300,
)
print("\n" + "=" * 60)
print("DECOMPOSED STEP 3: Prioritized Improvements (~300 tokens)")
print("=" * 60)
print(step3)

print("\n" + "=" * 60)
print("COMPARISON NOTES")
print("=" * 60)
print("• The one-shot response tries to cover everything but may be shallow or miss details.")
print("• Each decomposed step can go deeper because the model focuses on ONE aspect at a time.")
print("• The decomposed approach uses ~900 tokens total but each step only needs ~300 right.")


### How Focused Context Helps Each Step

Beyond the probability argument, decomposition provides a **cognitive** benefit to the model. When a model receives a prompt for "analyze architecture, security, AND performance," it must constantly switch between different analytical frameworks within a single generation. This is analogous to a human trying to write about three different topics simultaneously — quality suffers.

With decomposition, each subtask prompt provides **fresh, focused context**:
- The system prompt sets the expert role ("You are a security engineer")
- The user prompt asks about exactly one thing
- Any context from prior steps is explicitly labeled

The model doesn't need to "remember" to switch topics — each step starts clean with a clear objective. This is why decomposed results often have more depth on each dimension, even if the total token count is similar.


## Breaking Down Complex Tasks

Let's start with a complex task and break it down into subtasks. We'll use the example of analyzing a company's financial health.

In [ ]:
complex_task = """
Analyze the financial health of a company based on the following data:
- Revenue: $10 million
- Net Income: $2 million
- Total Assets: $15 million
- Total Liabilities: $7 million
- Cash Flow from Operations: $3 million
"""

decomposition_prompt = f"""
Break down the task of analyzing a company's financial health into 3 subtasks. \
For each subtask, provide a brief description of what it should accomplish.

Task: {complex_task}

Subtasks:
1.
"""

messages = [
    {"role": "system", "content": "You are a financial analyst."},
    {"role": "user", "content": decomposition_prompt},
]
subtasks = generate(messages)
print(subtasks)

## Information Flow Between Steps

In a decomposition pipeline, information flows through a chain of text-to-text transformations. Understanding how this flow works — and where it can break down — is essential for designing robust pipelines.


In [ ]:
# 4-step pipeline: Research → Outline → Draft → Review
# Watch how information transforms at each step

topic = "the impact of remote work on software team productivity"

# Step 1: Research key points
step1_research = run_subtask(
    "You are a research analyst. Be specific with data and findings.",
    f"List the 5 most important findings about {topic}. Include specific data points where possible.",
    max_new_tokens=300,
)
print("STEP 1 — Research Key Points:")
print(step1_research)
print(f"\n[Output: {len(step1_research)} chars, ~{len(step1_research)//4} tokens]")
print("=" * 50)

# Step 2: Create outline from research
step2_outline = run_subtask(
    "You are a technical writer creating a report outline.",
    f"Create a structured outline for a report on {topic}. Use the research findings to organize sections logically.",
    context=step1_research,
    max_new_tokens=300,
)
print("\nSTEP 2 — Structured Outline:")
print(step2_outline)
print(f"\n[Output: {len(step2_outline)} chars, ~{len(step2_outline)//4} tokens]")
print("=" * 50)

# Step 3: Draft introduction from outline
step3_draft = run_subtask(
    "You are a technical writer drafting a report.",
    f"Write a compelling introduction paragraph (150-200 words) for a report on {topic}. Ground it in the research findings.",
    context=f"Research:\n{step1_research}\n\nOutline:\n{step2_outline}",
    max_new_tokens=300,
)
print("\nSTEP 3 — Draft Introduction:")
print(step3_draft)
print(f"\n[Output: {len(step3_draft)} chars, ~{len(step3_draft)//4} tokens]")
print("=" * 50)

# Step 4: Review and improve
step4_review = run_subtask(
    "You are an editor reviewing technical writing for accuracy and clarity.",
    "Review this introduction. Check that claims match the research data. Improve clarity and flow. Output the revised introduction.",
    context=f"Research findings:\n{step1_research}\n\nDraft introduction:\n{step3_draft}",
    max_new_tokens=300,
)
print("\nSTEP 4 — Reviewed & Improved Introduction:")
print(step4_review)
print(f"\n[Output: {len(step4_review)} chars, ~{len(step4_review)//4} tokens]")
print("=" * 50)

# Show the transformation chain
print("\n📊 INFORMATION FLOW SUMMARY:")
print(f"  Step 1 (Research):  {len(step1_research):>5} chars → raw findings")
print(f"  Step 2 (Outline):   {len(step2_outline):>5} chars → structured organization")
print(f"  Step 3 (Draft):     {len(step3_draft):>5} chars → prose narrative")
print(f"  Step 4 (Review):    {len(step4_review):>5} chars → polished output")
print("\nNotice how each step TRANSFORMS the information, not just passes it along.")


In [ ]:
# Demonstrate context window pressure when chaining steps

# Step 1: Generate a deliberately verbose output
verbose_analysis = run_subtask(
    "You are a thorough researcher. Be extremely detailed and comprehensive.",
    "List everything important about renewable energy technologies: types, efficiency ratings, "
    "costs per kWh, current market trends, advantages, disadvantages, and future outlook "
    "for EACH major technology (solar, wind, hydro, nuclear, geothermal, biomass).",
    max_new_tokens=800,
)
print(f"VERBOSE output: {len(verbose_analysis)} chars (~{len(verbose_analysis)//4} tokens)")
print(f"Preview: {verbose_analysis[:300]}...")
print("=" * 50)

# Step 2a: Feed verbose output into a summary step
summary_from_verbose = run_subtask(
    "You are a concise executive summarizer. Be brief.",
    "Identify the top 3 most promising renewable technologies. Give exactly 3 bullet points.",
    context=verbose_analysis,
    max_new_tokens=200,
)
print(f"\nSUMMARY from verbose context ({len(summary_from_verbose)} chars):")
print(summary_from_verbose)
print("=" * 50)

# Compare: what if Step 1 was focused instead?
focused_analysis = run_subtask(
    "You are an energy analyst. Be concise — one sentence per technology.",
    "Rank the top 3 most promising renewable energy technologies. For each, state the key advantage and current cost per kWh.",
    max_new_tokens=200,
)
summary_from_focused = run_subtask(
    "You are a concise executive summarizer. Be brief.",
    "Summarize these findings in exactly 3 bullet points.",
    context=focused_analysis,
    max_new_tokens=200,
)
print(f"\nSUMMARY from focused context ({len(summary_from_focused)} chars):")
print(summary_from_focused)

print("\n" + "=" * 50)
print("💡 KEY TAKEAWAY: Context window pressure")
print(f"   Verbose context: {len(verbose_analysis)} chars → model must sift through noise")
print(f"   Focused context: {len(focused_analysis)} chars → model gets signal directly")
print("   Each step's output should contain ONLY what the next step needs.")


### The Information Bottleneck at Each Handoff

A critical concept in decomposition pipelines is the **information bottleneck** at each step transition. When step N produces output that feeds into step N+1:

1. **Only text survives**: Step N+1 sees the *text output* of step N, not the internal representations, attention patterns, or "reasoning" the model used internally. If step N "understood" something but didn't write it down, that understanding is lost.

2. **Information can be lost in translation**: If step 1 discovers 10 important facts but the outline in step 2 only references 7 of them, the draft in step 3 will never see those 3 missing facts — they're gone from the pipeline.

3. **Context window is finite**: Every character from previous steps consumes tokens from the model's context window, leaving fewer tokens for the current step's reasoning. Verbose intermediate outputs create "context pressure" that can degrade downstream quality.

**Design principle**: Each step should produce output that is **information-dense and appropriately scoped** for what the next step needs. Think of each handoff as passing a document to a new colleague — include what they need, leave out what they don't.


## Chaining Subtasks in Prompts

Now that we have our subtasks, let's create individual prompts for each and chain them together.

In [ ]:
SYSTEM_PROMPT = "You are a financial analyst specializing in company health assessments."

def analyze_profitability(revenue, net_income):
    """Analyze the company's profitability."""
    prompt = (
        f"Analyze the company's profitability based on the following data:\n"
        f"- Revenue: {revenue} million\n"
        f"- Net Income: {net_income} million\n\n"
        f"Calculate the profit margin and provide a brief analysis of the company's profitability."
    )
    return run_subtask(SYSTEM_PROMPT, prompt)

def analyze_liquidity(total_assets, total_liabilities, context=None):
    """Analyze the company's liquidity, optionally building on prior context."""
    prompt = (
        f"Analyze the company's liquidity based on the following data:\n"
        f"- Total Assets: {total_assets} million\n"
        f"- Total Liabilities: {total_liabilities} million\n\n"
        f"Calculate the current ratio and provide a brief analysis of the company's liquidity."
    )
    return run_subtask(SYSTEM_PROMPT, prompt, context=context)

def analyze_cash_flow(cash_flow, context=None):
    """Analyze the company's cash flow, optionally building on prior context."""
    prompt = (
        f"Analyze the company's cash flow based on the following data:\n"
        f"- Cash Flow from Operations: {cash_flow} million\n\n"
        f"Provide a brief analysis of the company's cash flow health."
    )
    return run_subtask(SYSTEM_PROMPT, prompt, context=context)

# Run the chained subtasks — each step builds on the previous one
profitability_analysis = analyze_profitability(10, 2)
liquidity_analysis = analyze_liquidity(15, 7, context=profitability_analysis)
cash_flow_analysis = analyze_cash_flow(3, context=f"{profitability_analysis}\n{liquidity_analysis}")

print("Profitability Analysis:\n", profitability_analysis)
print("\nLiquidity Analysis:\n", liquidity_analysis)
print("\nCash Flow Analysis:\n", cash_flow_analysis)

## Error Propagation Analysis

What happens when an error occurs partway through a decomposition chain? Understanding error propagation is crucial for building reliable pipelines.


In [ ]:
# 5-step chain with an intentional subtle error introduced at step 2
# We give the model contradictory data and see if errors propagate or self-correct

# Step 1: Extract data (correct input)
step1 = run_subtask(
    "You are a data analyst. Extract key metrics precisely.",
    """Extract and organize the key metrics from this quarterly report:

Company XYZ Q3 Report: Revenue $50M (up 12% YoY), Operating costs $35M,
Net income $15M, Employee count 500, Customer base 10,000, Customer churn 3.2%.""",
    max_new_tokens=200,
)
print("STEP 1 — Data Extraction (correct input):")
print(step1)
print("=" * 50)

# Step 2: Calculate growth — INTENTIONAL ERROR: we provide previous year revenue
# as $45M, but 12% growth from $45M = $50.4M, NOT $50M. This is an inconsistency.
step2 = run_subtask(
    "You are a financial analyst. Calculate growth rates precisely.",
    """Calculate year-over-year growth rates for each metric:
Current year: Revenue $50M, Net income $15M, Customers 10,000
Previous year: Revenue $45M, Net income $11M, Customers 8,500
Note: The company's press release states revenue grew 12% YoY.""",
    context=step1,
    max_new_tokens=300,
)
print("\nSTEP 2 — Growth Calculation (CONTAINS SUBTLE ERROR: $45M→$50M = 11.1%, not 12%):")
print(step2)
print("=" * 50)

# Step 3: Trend analysis based on step 2 output
step3 = run_subtask(
    "You are a business strategist analyzing growth trends.",
    "Analyze these growth trends. Which metrics show the strongest momentum? Are trends accelerating or decelerating?",
    context=step2,
    max_new_tokens=300,
)
print("\nSTEP 3 — Trend Analysis (does it propagate or catch the error?):")
print(step3)
print("=" * 50)

# Step 4: Risk assessment
step4 = run_subtask(
    "You are a risk analyst. Be thorough and skeptical.",
    "Identify the top 3 risks for this company. Consider data quality and consistency issues.",
    context=f"Growth analysis:\n{step2}\n\nTrend analysis:\n{step3}",
    max_new_tokens=300,
)
print("\nSTEP 4 — Risk Assessment:")
print(step4)
print("=" * 50)

# Step 5: Executive recommendation
step5 = run_subtask(
    "You are a management consultant. Base recommendations on specific numbers.",
    "Give a final recommendation: Buy, Hold, or Sell. Justify with specific numbers from the analysis chain.",
    context=f"Trends:\n{step3}\n\nRisks:\n{step4}",
    max_new_tokens=300,
)
print("\nSTEP 5 — Executive Recommendation:")
print(step5)

print("\n" + "=" * 50)
print("⚠️  ERROR TRACKING:")
print("   The subtle error: 12% growth from $45M should give $50.4M, not $50M.")
print("   Check above: did the model notice? Did it propagate into the recommendation?")
print("   This is why verification steps (like Step 4's 'data quality' check) matter.")


In [ ]:
# Run the error-prone step 3 times to observe variability in error handling

print("Running 3 trials of the same error-prone chain...\n")
trial_results = []

for trial_num in range(1, 4):
    print(f"{'='*60}")
    print(f"TRIAL {trial_num}")
    print(f"{'='*60}")

    # Step with the subtle error (same every time)
    s2 = run_subtask(
        "You are a financial analyst. Calculate precisely and show your work.",
        """Calculate year-over-year growth rates:
Current: Revenue $50M, Net income $15M
Previous: Revenue $45M, Net income $11M
Note: The press release claims revenue grew 12% YoY. Verify this.""",
        max_new_tokens=250,
    )

    # Verification step — can the model catch the inconsistency?
    s3 = run_subtask(
        "You are an auditor who checks financial calculations for accuracy.",
        "Review the growth rate calculations below. Flag ANY inconsistencies between stated and calculated values.",
        context=s2,
        max_new_tokens=250,
    )

    # Check if the error was caught
    response_lower = s3.lower()
    caught = any(phrase in response_lower for phrase in [
        "inconsisten", "mismatch", "not match", "incorrect", "discrepan",
        "does not equal", "doesn't equal", "11.1", "50.4", "not 12",
    ])

    trial_results.append(caught)
    status = "✅ CAUGHT the inconsistency" if caught else "❌ MISSED the inconsistency"
    print(f"  Result: {status}")
    print(f"  Auditor response preview: {s3[:250]}...\n")

caught_count = sum(trial_results)
print("=" * 60)
print(f"RESULTS: {caught_count}/3 trials caught the data inconsistency")
print("=" * 60)
if caught_count == 3:
    print("The model reliably catches this type of error — good!")
elif caught_count > 0:
    print("The model sometimes catches the error — verification steps help but aren't guaranteed.")
else:
    print("The model missed the error every time — this shows why human review of intermediate steps matters.")
print("\nThis variability is inherent to LLM pipelines. Mitigation strategies:")
print("  1. Add explicit verification steps that re-check calculations")
print("  2. Run critical chains multiple times and compare outputs")
print("  3. Include 'check your work' instructions in prompts")


### Understanding Error Cascades and Self-Correction

Errors in a decomposition chain can follow two patterns:

**Error Cascade** (step 2 wrong → step 3 wrong → step 4 wrong):
This happens when downstream steps blindly trust upstream outputs. If step 2 says "revenue grew 12%" and step 3 uses that number without checking, the error propagates. Each downstream step may even amplify the error by building conclusions on top of it.

**Self-Correction** (step 2 wrong → step 3 catches it):
This can happen when a downstream step has enough independent context to notice something is off. For example, if step 3 has access to the raw numbers AND the calculated growth rate, it might notice they don't match.

**Design implications for robust pipelines**:
1. **Include verification steps**: After any calculation or extraction step, add a step that explicitly checks the work
2. **Provide raw data to multiple steps**: Don't make step 5 rely entirely on step 4's summary — give it access to earlier outputs too
3. **Use "skeptical" system prompts**: Instructions like "verify all numbers" and "flag inconsistencies" make the model more likely to catch errors
4. **Run critical chains multiple times**: If the results agree, confidence is higher; if they diverge, investigate the disagreement


## Integrating Results

Finally, let's integrate the results from our subtasks to provide an overall analysis of the company's financial health.

In [ ]:
def integrate_results(profitability, liquidity, cash_flow):
    """Integrate the results from subtasks to provide an overall analysis."""
    combined_context = (
        f"Profitability Analysis:\n{profitability}\n\n"
        f"Liquidity Analysis:\n{liquidity}\n\n"
        f"Cash Flow Analysis:\n{cash_flow}"
    )
    prompt = "Summarize the key points and give an overall evaluation of the company's financial position."
    return run_subtask(
        "You are a senior financial analyst writing an executive summary.",
        prompt,
        context=combined_context,
        max_new_tokens=1024,
    )

overall_analysis = integrate_results(profitability_analysis, liquidity_analysis, cash_flow_analysis)
print("Overall Financial Health Analysis:\n", overall_analysis)

## When to Decompose vs. Ask Directly

Decomposition adds overhead: more prompts, more tokens, more latency. When is it actually worth it?


In [ ]:
import time

print("Comparing single-prompt vs. decomposed approaches at 3 difficulty levels\n")

# ── EASY: Simple factual question ─────────────────────────────
print("=" * 60)
print("EASY TASK: Simple factual question")
print("=" * 60)

t0 = time.time()
easy_single = generate([
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "What is the capital of France and approximately what is its population?"},
], max_new_tokens=100)
easy_single_time = time.time() - t0

t0 = time.time()
easy_d1 = run_subtask("You are a geography expert.", "What is the capital of France?", max_new_tokens=50)
easy_d2 = run_subtask("You are a demographics expert.", "What is the approximate population of Paris, France?", max_new_tokens=50)
easy_decomp_time = time.time() - t0

print(f"  Single prompt  ({easy_single_time:.1f}s): {easy_single.strip()[:200]}")
print(f"  Decomposed     ({easy_decomp_time:.1f}s): {easy_d1.strip()[:100]} | {easy_d2.strip()[:100]}")
print(f"  → Winner: SINGLE PROMPT (simpler, faster, no overhead needed)\n")

# ── MEDIUM: Multi-aspect analysis ─────────────────────────────
print("=" * 60)
print("MEDIUM TASK: Multi-aspect business analysis")
print("=" * 60)

medium_q = ("Should a mid-size software company (500 employees, $80M revenue) "
            "migrate from on-premise servers to cloud infrastructure? "
            "Consider costs, technical challenges, and team impact.")

t0 = time.time()
med_single = generate([
    {"role": "system", "content": "You are a technology strategy consultant."},
    {"role": "user", "content": medium_q},
], max_new_tokens=512)
med_single_time = time.time() - t0

t0 = time.time()
med_d1 = run_subtask("You are a cloud cost analyst.", f"Analyze ONLY the cost implications of: {medium_q}", max_new_tokens=200)
med_d2 = run_subtask("You are a cloud infrastructure architect.", f"Analyze ONLY the technical challenges of: {medium_q}", max_new_tokens=200)
med_d3 = run_subtask(
    "You are a technology strategy consultant.",
    "Synthesize these analyses into a clear recommendation.",
    context=f"Cost analysis:\n{med_d1}\n\nTechnical analysis:\n{med_d2}",
    max_new_tokens=200,
)
med_decomp_time = time.time() - t0

print(f"  Single prompt  ({med_single_time:.1f}s): {len(med_single)} chars")
print(f"  Decomposed     ({med_decomp_time:.1f}s): {len(med_d3)} chars (synthesis step)")
print(f"  → Roughly equal: both produce reasonable results for medium-complexity tasks\n")

# ── HARD: Multi-step numerical reasoning ──────────────────────
print("=" * 60)
print("HARD TASK: Multi-step numerical reasoning")
print("=" * 60)

hard_q = """A hospital system has 3 facilities:
- Facility A: 200 patients/day, 15% readmission rate
- Facility B: 350 patients/day, 8% readmission rate
- Facility C: 150 patients/day, 22% readmission rate
Total budget: $50M split proportionally by patient volume.
Propose a budget reallocation that minimizes the system-wide readmission rate.
Keep the total budget at $50M. Show specific numbers and calculations."""

t0 = time.time()
hard_single = generate([
    {"role": "system", "content": "You are a healthcare operations analyst. Show all calculations step by step."},
    {"role": "user", "content": hard_q},
], max_new_tokens=800)
hard_single_time = time.time() - t0

t0 = time.time()
h1 = run_subtask(
    "You are a data analyst. Show ALL calculations.",
    f"Calculate current budget per facility and total readmissions per facility.\n\n{hard_q}",
    max_new_tokens=300,
)
h2 = run_subtask(
    "You are a healthcare efficiency analyst. Use specific numbers.",
    "Calculate cost-per-readmission at each facility. Which facility gets the best and worst return on investment for reducing readmissions?",
    context=h1,
    max_new_tokens=300,
)
h3 = run_subtask(
    "You are a budget optimization specialist. Show exact dollar amounts.",
    "Propose a specific reallocation with exact dollar amounts. Calculate expected readmission improvements.",
    context=f"Current allocation:\n{h1}\n\nEfficiency analysis:\n{h2}",
    max_new_tokens=300,
)
h4 = run_subtask(
    "You are a healthcare strategy consultant. Verify all numbers add up.",
    "Summarize the reallocation strategy. Verify the total budget still equals $50M. State the expected impact on readmissions.",
    context=f"Allocation:\n{h1}\n\nEfficiency:\n{h2}\n\nProposal:\n{h3}",
    max_new_tokens=300,
)
hard_decomp_time = time.time() - t0

print(f"  Single prompt ({hard_single_time:.1f}s): {len(hard_single)} chars")
print(f"  First 400 chars: {hard_single[:400]}...\n")
print(f"  Decomposed ({hard_decomp_time:.1f}s): {len(h4)} chars (final synthesis)")
print(f"  First 400 chars: {h4[:400]}...\n")
print("  → Winner: DECOMPOSED — each step focuses on one calculation, reducing")
print("    errors in multi-step numerical reasoning. The single prompt often loses")
print("    track of intermediate calculations or makes arithmetic mistakes.")


### The Decomposition Decision Framework

Decomposition adds real overhead — more tokens generated, more API calls, higher latency. Use this framework to decide:

| Task Complexity | Single Prompt | Decomposed | Recommendation |
|---|---|---|---|
| **Simple** (factual, one-step) | ✅ Fast, accurate | ❌ Unnecessary overhead | Use single prompt |
| **Medium** (multi-aspect, qualitative) | ✅ Usually adequate | ✅ Slightly more thorough | Either works — prefer single for speed |
| **Hard** (multi-step reasoning, calculations) | ❌ Errors accumulate | ✅ Each step verified | Decompose — the quality gain outweighs the overhead |

**Rule of thumb**: Decompose when the task requires the model to maintain more than 2-3 "threads" of reasoning simultaneously, or when the task involves sequential calculations where earlier results feed into later ones.

**The overhead budget**: Each decomposition step costs roughly the same as a single prompt. A 4-step decomposition costs ~4× the tokens and ~4× the latency. This is worth it ONLY when the quality improvement compensates for the cost.


## Decomposition Patterns

Not all decomposition is linear. Different task structures call for different decomposition patterns. Here we explore three fundamental patterns: **Sequential**, **Parallel**, and **Recursive**.


In [ ]:
# PATTERN 1: Sequential — A → B → C → D
# Each step strictly depends on the previous step's output.
# Use when: tasks have a natural ordering where each step builds on the last.

print("=" * 60)
print("PATTERN: Sequential (A → B → C → D)")
print("Each step depends on the previous step's output.")
print("=" * 60)

# Example: Turning raw data into an actionable recommendation
raw_data = """Q1: Revenue $12M, Costs $10M, Customers 8,000
Q2: Revenue $14M, Costs $11M, Customers 9,200
Q3: Revenue $13M, Costs $12M, Customers 9,800
Q4: Revenue $16M, Costs $11.5M, Customers 11,000"""

seq_a = run_subtask(
    "You are a data analyst.",
    f"Calculate quarter-over-quarter growth rates for revenue, costs, and customers.\n\nData:\n{raw_data}",
    max_new_tokens=300,
)
print("A — Computed Growth Rates:")
print(seq_a[:300])
print()

seq_b = run_subtask(
    "You are a trend analyst.",
    "Identify the key trends. Is the company's trajectory improving or deteriorating?",
    context=seq_a,
    max_new_tokens=250,
)
print("B — Trend Identification (depends on A):")
print(seq_b[:300])
print()

seq_c = run_subtask(
    "You are a risk analyst.",
    "Based on these trends, what are the top 3 risks the company faces in the next 2 quarters?",
    context=f"Growth data:\n{seq_a}\n\nTrends:\n{seq_b}",
    max_new_tokens=250,
)
print("C — Risk Assessment (depends on A+B):")
print(seq_c[:300])
print()

seq_d = run_subtask(
    "You are a strategy consultant.",
    "Provide 3 specific, actionable recommendations to address these risks.",
    context=f"Trends:\n{seq_b}\n\nRisks:\n{seq_c}",
    max_new_tokens=250,
)
print("D — Recommendations (depends on B+C):")
print(seq_d[:300])

print("\n💡 Sequential is best when each step TRANSFORMS the information into a new form.")
print("   Raw data → Growth rates → Trends → Risks → Recommendations")


In [ ]:
# PATTERN 2: Parallel — A & B & C → D
# Independent analyses are run separately, then merged in a synthesis step.
# Use when: multiple independent aspects need analysis before combining.

print("=" * 60)
print("PATTERN: Parallel (A & B & C → D)")
print("Independent steps run separately, then merge.")
print("=" * 60)

product_desc = """A new mobile app for personal finance management targeting millennials.
Features: budget tracking, investment portfolio view, bill reminders, savings goals,
social spending comparisons. Monetization: freemium with $9.99/month premium tier.
Market: competing with Mint, YNAB, and Robinhood."""

# Three independent analyses (in a real pipeline, these could run concurrently)
par_a = run_subtask(
    "You are a market research analyst.",
    f"Analyze the competitive landscape and market opportunity for:\n{product_desc}",
    max_new_tokens=250,
)
print("A — Market Analysis (independent):")
print(par_a[:250])
print()

par_b = run_subtask(
    "You are a UX design expert.",
    f"Evaluate the feature set and user experience considerations for:\n{product_desc}",
    max_new_tokens=250,
)
print("B — UX Analysis (independent, parallel with A):")
print(par_b[:250])
print()

par_c = run_subtask(
    "You are a monetization strategist.",
    f"Evaluate the pricing model and revenue potential for:\n{product_desc}",
    max_new_tokens=250,
)
print("C — Monetization Analysis (independent, parallel with A & B):")
print(par_c[:250])
print()

# Synthesis step combines all three
par_d = run_subtask(
    "You are a startup advisor writing a go/no-go recommendation.",
    "Based on these three independent analyses, should this product be greenlit? Give a clear recommendation with the top 3 reasons.",
    context=f"Market Analysis:\n{par_a}\n\nUX Analysis:\n{par_b}\n\nMonetization Analysis:\n{par_c}",
    max_new_tokens=300,
)
print("D — Synthesis (depends on A + B + C):")
print(par_d[:400])

print("\n💡 Parallel is best when aspects are INDEPENDENT — the market analysis")
print("   doesn't need UX results, and vice versa. The synthesis step integrates them.")
print("   In production, A/B/C could run concurrently to reduce total latency.")


In [ ]:
# PATTERN 3: Recursive — Break → Solve subtasks → Combine
# The task is broken into subtasks, each solved independently, then results combined.
# Use when: the task is too complex to solve at once but has natural subdivisions.

print("=" * 60)
print("PATTERN: Recursive (Break → Solve parts → Combine)")
print("Break a complex task into parts, solve each, then combine.")
print("=" * 60)

complex_question = """Compare the software engineering practices of startups vs. 
large enterprises across these dimensions: development methodology, testing strategy, 
deployment practices, and team structure. For each dimension, explain the typical 
approach of each type and when one approach is better than the other."""

# Step 1: Break the task into subtasks
breakdown = run_subtask(
    "You are a task planner. Output ONLY the list of subtasks.",
    f"Break this question into independent sub-questions that can each be answered separately:\n\n{complex_question}",
    max_new_tokens=200,
)
print("BREAK — Task Decomposition:")
print(breakdown)
print()

# Step 2: Solve each subtask independently
subtask_results = []
dimensions = [
    ("development methodology", "Agile, Waterfall, hybrid approaches, sprint planning"),
    ("testing strategy", "unit tests, integration tests, QA processes, test coverage"),
    ("deployment practices", "CI/CD, release cycles, rollback strategies, feature flags"),
    ("team structure", "team size, roles, communication patterns, hierarchy"),
]

for dim_name, dim_hints in dimensions:
    result = run_subtask(
        f"You are a software engineering consultant specializing in {dim_name}.",
        f"Compare startups vs. large enterprises on {dim_name} ({dim_hints}). "
        f"Be specific about when each approach works best. Keep to 100 words.",
        max_new_tokens=200,
    )
    subtask_results.append((dim_name, result))
    print(f"SOLVE — {dim_name.title()}:")
    print(result[:250])
    print()

# Step 3: Combine results
combined_context = "\n\n".join(
    f"**{name.title()}**:\n{result}" for name, result in subtask_results
)
synthesis = run_subtask(
    "You are a senior engineering consultant writing an executive comparison.",
    "Synthesize these dimension-by-dimension comparisons into a coherent overall comparison. "
    "Highlight the 2-3 most important differences and provide a practical recommendation for "
    "a mid-size company (200 engineers) choosing between startup-style and enterprise-style practices.",
    context=combined_context,
    max_new_tokens=400,
)
print("COMBINE — Final Synthesis:")
print(synthesis)

print("\n💡 Recursive decomposition is best when a complex task has NATURAL SUBDIVISIONS")
print("   that can be solved independently. Each subtask is simpler and more focused,")
print("   and the combination step integrates them into a coherent whole.")


### Choosing the Right Decomposition Pattern

| Pattern | Structure | Best For | Latency | Example |
|---|---|---|---|---|
| **Sequential** | A → B → C → D | Tasks with natural ordering; each step transforms information | High (steps must run in order) | Data → Analysis → Risks → Recommendations |
| **Parallel** | A & B & C → D | Multi-aspect analysis with independent dimensions | Low (independent steps can run concurrently) | Market + UX + Pricing → Go/No-Go |
| **Recursive** | Break → Solve → Combine | Complex tasks with natural subdivisions | Medium (subtasks concurrent, but break/combine are sequential) | Compare X vs Y across N dimensions |

**Combining patterns**: Real-world pipelines often combine these. For example, a parallel pattern where one branch is itself a sequential chain:
```
A (research) → B (analysis)  ─┐
C (market data)               ├→ F (synthesis)
D (user surveys) → E (themes) ┘
```

The best decomposition mirrors the **natural structure of the problem**. If you find yourself forcing a problem into a pattern, reconsider whether decomposition is actually helping.
